# Week 3 Lab: The Weak Law of Large Numbers

## Learning Objectives

By the end of this lab, you will be able to:
1. Simulate random samples from the exponential distribution
2. Visualize how the sample mean converges to the population mean
3. Understand the role of variance in determining the rate of convergence

## Why This Matters for Econometrics

The Weak Law of Large Numbers (WLLN) is foundational to econometrics. It guarantees that our estimators—which are typically sample averages or functions thereof—converge in probability to the population parameters we care about. Without the WLLN, estimators like OLS would not be **consistent**, meaning they would not "zero in" on the true parameter value as sample size grows.

Today we work with simulated data (rather than real data) because simulation allows us to know the true population parameters. This lets us verify that the theory works as promised.

## The Exponential Distribution

We will work with data that is **exponentially distributed**. This distribution is useful for modelling waiting times, durations of unemployment spells, or time between events.

The pdf of the exponential distribution is:

$$
f(y; \mu) = 
\begin{cases}
\frac{1}{\mu} e^{-(y / \mu)} & \text{if } y \geq 0 \\
0 & \text{if } y < 0
\end{cases}
$$

A random variable with this pdf is called **exponentially distributed**, written $Y \sim \text{Exp}(\mu)$.

**Key properties:**
- $E(Y) = \mu$ (the mean equals the parameter)
- $\text{Var}(Y) = \mu^2$ (variance is the square of the mean)

This relationship between mean and variance—where higher means imply higher variance—is important for understanding convergence rates later.

## Loading Packages

We use three packages:
- `Distributions`: Provides probability distributions (Exponential, Normal, etc.)
- `Random`: Controls random number generation and seeds
- `Plots`: For visualization

In [ ]:
using Distributions
using Random
using Plots

## Exercise 1: Plot the PDF of the Exponential Distribution

Before working with random samples, let's visualize the population distribution itself.

**Task:** Plot the pdf of the exponential distribution for different values of $\mu$.

**Economic intuition:** If $Y$ represents unemployment duration, larger $\mu$ means longer expected unemployment spells *and* more variability in spell lengths. Notice how the pdf becomes more spread out (flatter) as $\mu$ increases.

### Worked Example: Plotting a Single PDF

Here's how to create an exponential distribution and plot its pdf:

In [ ]:
# Create an exponential distribution with μ = 0.5
μ = 0.5
d = Exponential(μ)  # Distribution object with mean μ

# Plot the pdf using an anonymous function x -> pdf(d, x)
plot(x -> pdf(d, x), 0, 5,
     label = "μ = $μ",
     xlabel = "y",
     ylabel = "f(y)",
     title = "PDF of Exponential Distribution")

### Your Turn: Plot PDFs for Different μ Values

Following the pattern above, create plots for $\mu = 1.0$ and $\mu = 1.5$.

In [ ]:
# Plot pdf for μ = 1.0
# Hint: Follow the same pattern as the worked example above

μ = nothing  # Set μ = 1.0
d = nothing  # Create Exponential distribution

# YOUR CODE HERE
# Create a plot similar to the example, with ylim = (0, 2)

In [ ]:
# Plot pdf for μ = 1.5

μ = nothing  # Set μ = 1.5
d = nothing  # Create Exponential distribution

# YOUR CODE HERE

### Challenge: Overlay Multiple PDFs

Create a single plot that overlays the pdfs for $\mu \in \{0.5, 1.0, 1.5, 3.0\}$.

**Hints:**
- Create an empty plot first with `p = plot(...)`
- Use a `for` loop to iterate over μ values
- Use `plot!(p, ...)` to add each curve to the existing plot

In [ ]:
# Overlay multiple pdfs to compare shapes
# This reveals how variance (spread) increases with μ

p = plot(xlabel = "y",
         ylabel = "f(y)",
         title = "PDFs of Exponential Distributions",
         legend = :topright)

# YOUR CODE HERE
# Loop over μ values [0.5, 1.0, 1.5, 3.0]
# Inside the loop: create the distribution, then add its pdf curve to p

p

**Interpretation:** Notice that:
- Smaller $\mu$ gives a distribution concentrated near zero with a steep decline
- Larger $\mu$ spreads the distribution out, giving more probability to larger values
- The mode is always at zero, but the mean shifts rightward as $\mu$ increases

## Exercise 2: Create a Random Sample and Compare to Population

Now let's draw a random sample and see how well sample statistics match population parameters.

**Task:** 
1. Set sample size $N = 5000$ and $\mu = 2$
2. Draw a random sample
3. Compare the sample mean and variance to the true values ($E(Y) = 2$, $\text{Var}(Y) = 4$)

**Key insight:** With a large sample, sample statistics should be close to population parameters. This is exactly what the WLLN promises.

In [ ]:
# Set parameters
N = 5000    # Sample size
μ = 2.0     # Population mean (and distribution parameter)

# Fix seed for reproducibility
# This ensures everyone gets the same "random" sample
Random.seed!(42)

# Create distribution and draw sample
d = nothing      # Create Exponential(μ) distribution
sample = nothing # Draw N independent observations using rand(d, N)

# YOUR CODE HERE

# After completing, uncomment the lines below to check your results:
# println("Population mean E(Y) = μ = ", μ)
# println("Sample mean ȳ = ", round(mean(sample), digits=4))
# println()
# println("Population variance Var(Y) = μ² = ", μ^2)
# println("Sample variance s² = ", round(var(sample), digits=4))

### Visual Comparison: Histogram vs Population PDF

Create side-by-side plots comparing:
1. A histogram of your sample (normalized to density)
2. The population pdf

**Hints:**
- Use `histogram(sample, normalize = true, ...)` for the density histogram
- Use `plot(x -> pdf(d, x), 0, 15, ...)` for the population pdf
- Use `plot(p1, p2, layout = (1, 2), size = (900, 400))` to combine plots

In [ ]:
# Visual comparison: histogram of sample vs population pdf

p1 = nothing  # Sample histogram (normalized to density)
p2 = nothing  # Population PDF curve

# YOUR CODE HERE
# Create p1 and p2 following the hints above, then combine them side by side

**Interpretation:** The sample histogram closely approximates the population pdf. The sample mean is very close to $\mu = 2$. This is the WLLN at work: with $N = 5000$, we're seeing the convergence in action.

## Exercise 3: Visualize the Law of Large Numbers

The WLLN states that for a random sample $Y_1, \ldots, Y_N$:

$$\bar{Y}_N \xrightarrow{p} E(Y_1) \quad \text{as } N \to \infty$$

where $\bar{Y}_N = \frac{1}{N}\sum_{i=1}^N Y_i$ is the sample mean.

**The idea:** As we accumulate more observations, the sample mean "settles down" to the population mean. Early on (small $N$), the sample mean fluctuates wildly. As $N$ grows, fluctuations diminish.

**Implementation approach:**
1. Draw one large "super-sample" of size $N_{\max}$
2. For each $n = 1, 2, \ldots, N_{\max}$, compute $\bar{Y}_n$ using the first $n$ observations
3. Plot $\bar{Y}_n$ against $n$

**Note:** The code below is *deliberately inefficient*—we'll optimize it next week. For now, focus on understanding the logic.

### Worked Example: WLLN Visualization

Here's the basic pattern for visualizing the WLLN:

In [ ]:
# Start with a small n_max to verify the code works
n_max = 100
μ = 2.0

# Pre-allocate vector to store running sample means
ȳ = Vector{Float64}(undef, n_max)

# Create distribution and draw super-sample
d = Exponential(μ)
Random.seed!(42)
super_sample = rand(d, n_max)

# Compute sample mean for each sample size n = 1, ..., n_max
# (This nested loop is inefficient—we'll fix this next week)
for n in 1:n_max
    sub_sample = super_sample[1:n]  # First n observations
    ȳ[n] = mean(sub_sample)
end

# Plot the convergence
plot(1:n_max, ȳ,
     label = "Sample mean ȳₙ",
     xlabel = "Sample size n",
     ylabel = "ȳₙ",
     title = "WLLN: Convergence of Sample Mean")
hline!([μ], linewidth = 2, linestyle = :dash, label = "μ = $μ")

**Observation:** Even with $n = 100$, the sample mean is already quite close to $\mu = 2$. Let's see what happens with smaller variance.

### Your Turn: Explore Different Parameters

Following the pattern above, create a WLLN visualization with:
1. $\mu = 0.5$ (smaller variance should mean faster convergence)
2. Then try $n_{\max} = 5000$ with $\mu = 2.0$ to see longer-run convergence

In [ ]:
# Smaller μ means smaller variance (Var = μ²)
# Convergence should be faster!

n_max = 100
μ = 0.5  # Variance is now only 0.25

ȳ = nothing  # Pre-allocate vector for running means
d = nothing  # Create Exponential distribution
super_sample = nothing  # Draw super_sample

Random.seed!(42)

# YOUR CODE HERE
# 1. Create the distribution and draw the super_sample
# 2. Loop through n = 1:n_max and compute running means
# 3. Plot with ylims = (0, 1.2)

**Key insight:** Lower variance leads to faster convergence. The fluctuations are smaller because individual observations don't deviate as much from the mean.

In [ ]:
# Now increase n_max to see longer-run convergence

n_max = 5000
μ = 2.0

ȳ = nothing
d = nothing
super_sample = nothing

Random.seed!(42)

# YOUR CODE HERE

**Interpretation:** The sample mean starts with large fluctuations when $n$ is small, but gradually "locks in" on the population mean $\mu = 2$ as $n$ grows. This is the WLLN in action!

## Exercise 4: Compare Convergence Rates

Let's systematically compare how the variance affects convergence speed.

**Theory:** The variance of the sample mean is $\text{Var}(\bar{Y}_N) = \text{Var}(Y)/N$. For the exponential distribution:

$$\text{Var}(\bar{Y}_N) = \frac{\mu^2}{N}$$

So larger $\mu$ means the sample mean has larger variance (more fluctuation) for any given sample size.

**Task:** Write a function `visualize_wlln` that takes a vector of μ values and n_max, and creates a plot comparing the convergence rates.

In [ ]:
"""
    visualize_wlln(μ_values::Vector{<:Real}, n_max::Integer)

Illustrate the Weak Law of Large Numbers for exponential distributions.

# Arguments
- `μ_values`: Vector of exponential distribution parameters to compare
- `n_max`: Maximum sample size for the simulation

# Returns
A plot showing convergence of sample means for each μ value.

# Details
For the exponential distribution, Var(Y) = μ², so larger μ implies
slower convergence (more fluctuation in ȳₙ for any given n).
"""
function visualize_wlln(μ_values::Vector{<:Real}, n_max::Integer)
    # Create the base plot
    p = plot(title = "Comparing Convergence Rates (WLLN)",
             xlabel = "Sample size n",
             ylabel = "Sample mean ȳₙ",
             legend = :right)
    
    # YOUR CODE HERE
    # Loop over μ_values. For each μ:
    #   - Reset the random seed for a fair comparison
    #   - Draw a super_sample and compute running means (as in Exercise 3)
    #   - Add the true mean as a dashed horizontal line
    #   - Plot the convergence path with a label showing μ and its variance
    
    error("Not yet implemented")
    
    return p
end

In [ ]:
# Test your function
# Uncomment when ready:

# visualize_wlln([0.1, 0.5, 1.0, 2.0], 1000)

## Summary

Today we demonstrated the Weak Law of Large Numbers:

1. **The WLLN guarantees consistency:** As sample size grows, the sample mean converges in probability to the population mean.

2. **Variance determines convergence speed:** Higher variance means the sample mean fluctuates more, requiring larger samples to achieve the same precision.

3. **Practical implications:** When estimating population parameters, having lower-variance data or larger samples leads to more reliable estimates.

**Next week:** We'll improve the efficiency of our simulation code and move on to the Central Limit Theorem, which tells us about the *distribution* of the sample mean, not just its limit.